# Daily Task Agent

A single-notebook version of the `daily_task_agent` package. The package
is a small NL-driven agent that parses user requests, routes them to
tool functions, and persists tasks/reminders to a JSON file.

**Features**

- Intent parsing: `add_task`, `set_reminder`, `list_tasks`, `delete_task`,
  `mark_done`, `clear`, `help`.
- Date/time extraction for phrases like `tomorrow at 7 PM`,
  `on Friday at 10 AM`, or `in 30 minutes`.
- JSON storage under `~/.daily_task_agent/tasks.json` (override with the
  `TASK_AGENT_HOME` environment variable).
- Pluggable tool registry so a real calendar/notification backend could
  be swapped in later.

Run the cells top-to-bottom, then try `agent.handle("...")` or call
`repl()` for an interactive session inside the notebook.


In [1]:
import json
import os
import re
from datetime import datetime, timedelta, time
from pathlib import Path
from threading import Lock
from typing import Any

try:
    import dateparser  # type: ignore
    _HAS_DATEPARSER = True
except Exception:  # pragma: no cover - fallback path
    _HAS_DATEPARSER = False

print(f"dateparser available: {_HAS_DATEPARSER}")


dateparser available: False


## Storage

JSON-file persistence for tasks and reminders.


In [2]:
_DATA_DIR = Path(os.environ.get("TASK_AGENT_HOME", Path.home() / ".daily_task_agent"))
_DATA_FILE = _DATA_DIR / "tasks.json"
_lock = Lock()


def _ensure_dir() -> None:
    _DATA_DIR.mkdir(parents=True, exist_ok=True)


def _load() -> dict:
    _ensure_dir()
    if not _DATA_FILE.exists():
        return {"tasks": []}
    with _DATA_FILE.open("r", encoding="utf-8") as f:
        return json.load(f)


def _save(data: dict) -> None:
    _ensure_dir()
    tmp = _DATA_FILE.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    tmp.replace(_DATA_FILE)


def get_all_tasks() -> list:
    with _lock:
        return _load().get("tasks", [])


def add_task(task: dict) -> dict:
    with _lock:
        data = _load()
        task["id"] = max([t.get("id", 0) for t in data["tasks"]], default=0) + 1
        data["tasks"].append(task)
        _save(data)
        return task


def delete_task(task_id: int) -> bool:
    with _lock:
        data = _load()
        before = len(data["tasks"])
        data["tasks"] = [t for t in data["tasks"] if t.get("id") != task_id]
        if len(data["tasks"]) == before:
            return False
        _save(data)
        return True


def set_done(task_id: int, done: bool = True) -> bool:
    """Mark a task done/undone in place. Returns False if the id is unknown."""
    with _lock:
        data = _load()
        for t in data.setdefault("tasks", []):
            if t.get("id") == task_id:
                t["done"] = done
                _save(data)
                return True
        return False


def clear_all() -> None:
    with _lock:
        _save({"tasks": []})


# Module-style namespace so the rest of the notebook can call storage.X().
class _StorageModule:
    get_all_tasks = staticmethod(get_all_tasks)
    add_task = staticmethod(add_task)
    delete_task = staticmethod(delete_task)
    set_done = staticmethod(set_done)
    clear_all = staticmethod(clear_all)

storage = _StorageModule()


## NLP — intent + entity parsing

Parses a free-form English sentence into an intent and a dict of
entities (title, due ISO string, priority, task id, when).


In [3]:
# ---- intent detection ----------------------------------------------------

_LIST_PATTERNS = [
    r"\bwhat\b.*\bi have\b",
    r"\bshow (me )?(my )?(tasks?|to-?dos?|reminders?)\b",
    r"\blist (my )?(tasks?|to-?dos?|reminders?)\b",
    r"\b(any )?tasks? (for |due )(today|tomorrow)\b",
    r"\btoday'?s tasks?\b",
    r"\btomorrow'?s tasks?\b",
    r"\bupcoming\b",
]

_REMIND_PATTERNS = [
    r"\bremind (me|us) to\b",
    r"\bset (a )?reminder\b",
    r"\bdon'?t let me forget to\b",
    r"\balert me to\b",
]

_ADD_PATTERNS = [
    r"\badd\b(?:\s+\w+){0,4}\s+task\b",
    r"\bcreate\b(?:\s+\w+){0,4}\s+task\b",
    r"\bnew task\b",
    r"\bi (need|have) to\b",
    r"\bi should\b",
    r"\bi must\b",
    r"\bi have a (meeting|appointment|call|class)\b",
]

_DELETE_PATTERNS = [
    r"\bdelete\b",
    r"\bremove\b",
    r"\bcancel\b",
]

_DONE_PATTERNS = [
    r"\bmark .* (as )?done\b",
    r"\bmark .* complete\b",
    r"\bi('?ve| have) (finished|completed|done)\b",
    r"\bfinish (the )?task\b",
]

_CLEAR_PATTERNS = [
    r"\bclear (all )?(my )?tasks\b",
    r"\breset (my )?tasks\b",
]

_HELP_PATTERNS = [r"\bhelp\b", r"\bwhat can you do\b"]


def _matches_any(text: str, patterns):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def detect_intent(text: str) -> str:
    t = text.lower().strip()
    if not t:
        return "unknown"
    if _matches_any(t, _HELP_PATTERNS):
        return "help"
    if _matches_any(t, _CLEAR_PATTERNS):
        return "clear"
    if _matches_any(t, _LIST_PATTERNS):
        return "list_tasks"
    if _matches_any(t, _DONE_PATTERNS):
        return "mark_done"
    if _matches_any(t, _DELETE_PATTERNS):
        return "delete_task"
    if _matches_any(t, _REMIND_PATTERNS):
        return "set_reminder"
    if _matches_any(t, _ADD_PATTERNS):
        return "add_task"
    if _extract_datetime(t) and ("remind" in t or " at " in t or " on " in t
                                or " tomorrow" in t or " today" in t):
        return "set_reminder"
    return "unknown"


In [4]:
# ---- entity extraction ---------------------------------------------------

_WHEN_KEYWORDS = {
    "today": "today",
    "tonight": "today",
    "tomorrow": "tomorrow",
    "yesterday": "yesterday",
}


def _extract_when(text: str):
    for k, v in _WHEN_KEYWORDS.items():
        if re.search(rf"\b{k}\b", text, flags=re.IGNORECASE):
            return v
    return None


def _extract_priority(text: str):
    m = re.search(r"\b(low|medium|high|urgent)\s*priority\b", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).lower()
    if re.search(r"\b(urgent|asap|important)\b", text, flags=re.IGNORECASE):
        return "high"
    return None


def _extract_task_id(text: str):
    m = re.search(r"\b(?:task\s*)?#?(\d+)\b", text)
    if m:
        return int(m.group(1))
    return None


def _strip_intro_phrase(text: str) -> str:
    """Remove imperative prefixes so the remaining text is closer to the title."""
    patterns = [
        r"^\s*remind me to\s+",
        r"^\s*set a reminder to\s+",
        r"^\s*set a reminder for\s+",
        r"^\s*alert me to\s+",
        r"^\s*add (?:a |an )?(?:\w+\s+){0,3}?task (?:to |for )?\s*",
        r"^\s*create (?:a |an )?(?:\w+\s+){0,3}?task (?:to |for )?\s*",
        r"^\s*new task:?\s+",
        r"^\s*i (need|have) to\s+",
        r"^\s*i should\s+",
        r"^\s*i must\s+",
        r"^\s*don'?t let me forget to\s+",
    ]
    for p in patterns:
        text = re.sub(p, "", text, flags=re.IGNORECASE).strip()
    return text


def _clean_title(title: str) -> str:
    title = re.sub(r"\s+", " ", title).strip().rstrip(".?!")
    title = re.sub(
        r"[\s,]+(?:at|on|by|in|before|after)$", "", title, flags=re.IGNORECASE
    ).strip()
    return title.title()


In [5]:
# ---- date/time extraction ------------------------------------------------

def _extract_datetime(text: str):
    """Find the first date/time mentioned in the text and parse it."""
    if _HAS_DATEPARSER:
        settings = {
            "PREFER_DATES_FROM": "future",
            "RELATIVE_BASE": datetime.now(),
            "RETURN_AS_TIMEZONE_AWARE": False,
        }
        candidates = re.findall(
            r"(?:tomorrow|today|tonight|"
            r"(?:mon|tues|wednes|thurs|fri|satur|sun)day|"
            r"\d{1,2}(?::\d{2})?\s*(?:am|pm)?|"
            r"\d{1,2}/\d{1,2}(?:/\d{2,4})?|"
            r"\d{4}-\d{2}-\d{2}|"
            r"in \d+ (?:minute|hour|day|week)s?)(?:\s+at\s+\d{1,2}(?::\d{2})?\s*(?:am|pm)?)?",
            text,
            flags=re.IGNORECASE,
        )
        for c in candidates:
            dt = dateparser.parse(c, settings=settings)
            if dt:
                return dt
        return dateparser.parse(text, settings=settings)
    return _fallback_datetime(text)


def _fallback_datetime(text: str):
    now = datetime.now()
    t = text.lower()
    base_date = now.date()
    if "tomorrow" in t:
        base_date = (now + timedelta(days=1)).date()
    elif "today" in t or "tonight" in t:
        base_date = now.date()
    m = re.search(r"\b(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\b", t)
    if not m:
        return datetime.combine(base_date, time()) if (base_date != now.date() or "today" in t or "tomorrow" in t) else None
    hour = int(m.group(1))
    minute = int(m.group(2) or 0)
    ampm = m.group(3)
    if ampm:
        if ampm == "pm" and hour < 12:
            hour += 12
        if ampm == "am" and hour == 12:
            hour = 0
    elif hour <= 23 and "tomorrow" in t and hour < 12:
        hour += 12
    if not (0 <= hour <= 23 and 0 <= minute <= 59):
        return datetime.combine(base_date, time())
    return datetime.combine(base_date, time(hour, minute))


def _remove_datetime_phrases(text: str) -> str:
    """Strip date/time substrings out of a sentence to leave a clean title."""
    patterns = [
        r"\b(?:at\s+)?\d{1,2}:\d{2}\s*(?:am|pm)?\b",
        r"\b(?:at\s+)?\d{1,2}\s*(?:am|pm)\b",
        r"\b(?:at\s+)?\d{1,2}\s*o'?clock\b",
        r"\b\d{1,2}/\d{1,2}(?:/\d{2,4})?\b",
        r"\b\d{4}-\d{2}-\d{2}\b",
        r"\b(?:in|after)\s+\d+\s+(?:minute|hour|day|week)s?\b",
        r"\b(?:tomorrow|today|tonight|yesterday)\b",
        r"\bat\s+\d{1,2}(?::\d{2})?\s*(?:am|pm)?\b",
        r"\b(?:on|by|before|after)\s+(?:mon|tues|wednes|thurs|fri|satur|sun)day\b",
        r"\b(?:this|next)\s+(?:mon|tues|wednes|thurs|fri|satur|sun)day\b",
        r"\b(?:low|medium|high|urgent)\s*priority\b",
        r"\b(urgent|asap|important)\b",
    ]
    out = text
    for p in patterns:
        out = re.sub(p, "", out, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", out).strip(" ,.-")


In [6]:
# ---- public API ----------------------------------------------------------

def parse(text: str) -> dict:
    intent = detect_intent(text)
    when = _extract_when(text)
    priority = _extract_priority(text)
    task_id = _extract_task_id(text)

    when_for_list = when
    if intent == "list_tasks" and when_for_list is None:
        if re.search(r"\b(today|now|tonight)\b", text, flags=re.IGNORECASE):
            when_for_list = "today"

    dt = _extract_datetime(text)
    due_iso = dt.isoformat(timespec="minutes") if dt else None

    title = _clean_title(_remove_datetime_phrases(_strip_intro_phrase(text)))

    return {
        "intent": intent,
        "entities": {
            "title": title or None,
            "when": when_for_list,
            "time": dt.strftime("%H:%M") if dt else None,
            "priority": priority,
            "task_id": task_id,
            "due": due_iso,
        },
    }


# Module-style namespace so the rest of the notebook can call nlp.parse().
class _NlpModule:
    parse = staticmethod(parse)
    detect_intent = staticmethod(detect_intent)

nlp = _NlpModule()

# Sanity check: parse the two prompt examples.
print(nlp.parse("Remind me to study DSA tomorrow at 7 PM."))
print(nlp.parse("What tasks do I have today?"))


{'intent': 'set_reminder', 'entities': {'title': 'Study Dsa', 'when': 'tomorrow', 'time': '19:00', 'priority': None, 'task_id': 7, 'due': '2026-09-01T19:00'}}
{'intent': 'list_tasks', 'entities': {'title': 'What Tasks Do I Have', 'when': 'today', 'time': '00:00', 'priority': None, 'task_id': None, 'due': '2026-08-31T00:00'}}


## Tools

Tool functions the agent can call to manage tasks and reminders.


In [7]:
VALID_PRIORITIES = {"low", "medium", "high"}


def _coerce_priority(value):
    if value is None:
        return "medium"
    p = str(value).strip().lower()
    return p if p in VALID_PRIORITIES else "medium"


def add_task(title: str, due=None, priority: str = "medium", notes=None) -> dict:
    """Create a new task. `due` is an ISO-8601 string."""
    if not title or not title.strip():
        raise ValueError("Task title cannot be empty.")

    task = {
        "title": title.strip(),
        "due": due,
        "priority": _coerce_priority(priority),
        "notes": (notes or "").strip() or None,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "done": False,
    }
    return storage.add_task(task)


def list_tasks(when=None, only_open: bool = True) -> list:
    """Return tasks. `when` in {None, 'today', 'tomorrow', 'overdue', 'all'}."""
    tasks = storage.get_all_tasks()
    if only_open:
        tasks = [t for t in tasks if not t.get("done")]

    if when in (None, "all"):
        return tasks

    today = datetime.now().date()
    tomorrow = today + timedelta(days=1)

    def matches(t: dict) -> bool:
        due_str = t.get("due")
        if not due_str:
            return False
        try:
            due_dt = datetime.fromisoformat(due_str)
        except ValueError:
            return False
        d = due_dt.date()
        if when == "today":
            return d == today
        if when == "tomorrow":
            return d == tomorrow
        if when == "overdue":
            return d < today
        return False

    return [t for t in tasks if matches(t)]


def delete_task(task_id: int) -> bool:
    return storage.delete_task(int(task_id))


def set_reminder(title: str, remind_at: str, message=None) -> dict:
    """A reminder is a task with a due time and a reminder-style note."""
    return add_task(
        title=title.strip(),
        due=remind_at,
        priority="high",
        notes=message or f"Reminder: {title.strip()}",
    )


def mark_done(task_id: int) -> bool:
    return storage.set_done(int(task_id), True)


TOOL_REGISTRY = {
    "add_task": add_task,
    "list_tasks": list_tasks,
    "delete_task": delete_task,
    "set_reminder": set_reminder,
    "mark_done": mark_done,
}


## Agent

Routes parsed intents to tools and formats user-facing replies.


In [8]:
HELP_TEXT = """\
I can help you manage your daily tasks. Try things like:

  - "Remind me to study DSA tomorrow at 7 PM"
  - "What tasks do I have today?"
  - "Add a task to call the dentist on Friday at 10 AM"
  - "Show my tasks"
  - "Mark task #2 as done"
  - "Delete task #3"
  - "Clear all tasks"
"""


def _format_due(due):
    if not due:
        return "no due date"
    try:
        dt = datetime.fromisoformat(due)
        return dt.strftime("%a %b %d, %Y at %I:%M %p").lstrip("0")
    except ValueError:
        return due


def _format_task(t: dict) -> str:
    marker = "\u2713" if t.get("done") else "\u2022"
    due = _format_due(t.get("due"))
    prio = t.get("priority", "medium")
    return f"  {marker} #{t.get('id'):<3} [{prio:<6}] {t.get('title'):<40} (due: {due})"


def _reply_list(tasks: list, label: str) -> str:
    if not tasks:
        return f"You have no {label}."
    lines = [f"Here are your {label} ({len(tasks)}):", ""]
    for t in tasks:
        lines.append(_format_task(t))
    return "\n".join(lines)


def handle(text: str) -> str:
    """Top-level entry point. Takes a user sentence, returns a reply string."""
    parsed = nlp.parse(text)
    intent = parsed["intent"]
    e = parsed["entities"]

    if intent == "help":
        return HELP_TEXT

    if intent == "unknown":
        return ("I didn't understand that. " + HELP_TEXT).strip()

    if intent == "clear":
        storage.clear_all()
        return "Cleared all your tasks."

    if intent == "list_tasks":
        when = e.get("when") or "all"
        if when == "yesterday":
            return "I can only show today, tomorrow, or overdue tasks."
        tasks = list_tasks(when=when)
        label = {
            "today": "tasks for today",
            "tomorrow": "tasks for tomorrow",
            "overdue": "overdue tasks",
            "all": "open tasks",
        }.get(when, "open tasks")
        return _reply_list(tasks, label)

    if intent == "add_task":
        title = e.get("title")
        if not title:
            return "What should I add? Please describe the task."
        task = add_task(
            title=title,
            due=e.get("due"),
            priority=e.get("priority") or "medium",
        )
        when_str = _format_due(task.get("due"))
        return (
            f"Got it - added task #{task['id']}: \"{task['title']}\" "
            f"(due: {when_str}, priority: {task['priority']})."
        )

    if intent == "set_reminder":
        title = e.get("title")
        if not title:
            return "What should I remind you about?"
        if not e.get("due"):
            return "When should I remind you? Please include a date or time."
        reminder = set_reminder(
            title=title,
            remind_at=e["due"],
            message=f"Reminder: {title}",
        )
        when_str = _format_due(reminder.get("due"))
        return f"Reminder set: \"{reminder['title']}\" at {when_str} (task #{reminder['id']})."

    if intent == "delete_task":
        task_id = e.get("task_id")
        if not task_id:
            title = e.get("title")
            if title:
                for t in list_tasks(when="all"):
                    if title.lower() in t["title"].lower():
                        task_id = t["id"]
                        break
        if not task_id:
            return "Which task should I delete? Try: 'delete task #3' or 'delete <title>'."
        if delete_task(task_id):
            return f"Deleted task #{task_id}."
        return f"I couldn't find task #{task_id}."

    if intent == "mark_done":
        task_id = e.get("task_id")
        if not task_id:
            title = e.get("title")
            if title:
                for t in list_tasks(when="all"):
                    if title.lower() in t["title"].lower():
                        task_id = t["id"]
                        break
        if not task_id:
            return "Which task did you finish? Try: 'mark task #2 as done'."
        if mark_done(task_id):
            return f"Nice - task #{task_id} marked as done."
        return f"I couldn't find task #{task_id}."

    return "Sorry, I couldn't handle that request."


# Module-style namespace so the rest of the notebook can call agent.handle().
class _AgentModule:
    handle = staticmethod(handle)

agent = _AgentModule()


## Demo

Try the two prompt examples, plus a few extras.


In [9]:
# Start clean.
storage.clear_all()

print(agent.handle("Remind me to study DSA tomorrow at 7 PM."))
print()
print(agent.handle("Add a task to call mom today at 6 PM."))
print()
print(agent.handle("Remind me to pay rent tomorrow at 9 AM."))
print()
print(agent.handle("What tasks do I have today?"))
print()
print(agent.handle("Add a high priority task to submit taxes on Friday."))
print()
print(agent.handle("Mark task #1 as done"))
print()
print(agent.handle("Show my tasks"))
print()
print(agent.handle("help"))


Reminder set: "Study Dsa" at Tue Sep 01, 2026 at 07:00 PM (task #1).

Got it - added task #2: "Call Mom" (due: Mon Aug 31, 2026 at 06:00 PM, priority: medium).

Reminder set: "Pay Rent" at Tue Sep 01, 2026 at 09:00 AM (task #3).

Here are your tasks for today (1):

  • #2   [medium] Call Mom                                 (due: Mon Aug 31, 2026 at 06:00 PM)

Got it - added task #4: "Submit Taxes" (due: no due date, priority: high).

Nice - task #1 marked as done.

Here are your open tasks (3):

  • #2   [medium] Call Mom                                 (due: Mon Aug 31, 2026 at 06:00 PM)
  • #3   [high  ] Pay Rent                                 (due: Tue Sep 01, 2026 at 09:00 AM)
  • #4   [high  ] Submit Taxes                             (due: no due date)

I can help you manage your daily tasks. Try things like:

  - "Remind me to study DSA tomorrow at 7 PM"
  - "What tasks do I have today?"
  - "Add a task to call the dentist on Friday at 10 AM"
  - "Show my tasks"
  - "Mark task #

## Interactive REPL

A notebook-friendly REPL. Run the cell, then type requests in the input
box. Type `quit`, `exit`, or `q` to leave; `list` / `ls` / `show` to
list open tasks; `reset` / `wipe` to clear everything.


In [10]:
def repl() -> int:
    print("=" * 60)
    print("  Daily Task Agent - type 'help' or 'quit'")
    print("=" * 60)
    while True:
        try:
            user_input = input("\n> ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            return 0
        if not user_input:
            continue
        if user_input.lower() in {"quit", "exit", "q"}:
            print("Goodbye!")
            return 0
        if user_input.lower() in {"list", "ls", "show"}:
            user_input = "show my tasks"
        if user_input.lower() in {"reset", "wipe"}:
            user_input = "clear all my tasks"
        print(agent.handle(user_input))

# Uncomment to run the interactive REPL inside the notebook:
repl()


  Daily Task Agent - type 'help' or 'quit'
Goodbye!


0